# Tennessee Eastman Process — Modellbildung

Lädt die Rohdaten `data_train`/`data_test`/`data_error` aus *tep_generation*, trainiert/lädt ein MLP auf den stationären Trainingsphasen, wertet den Prädiktionsfehler auf Test- und Fehlerfalldaten aus und speichert die gescorten Artefakte `data_scored_test`/`data_scored_error` für *tep_detection* und *tep_adaptation*.

In [ ]:
import os
from pathlib import Path

work_dir = os.getcwd()
DEFAULT_BASE_DIR = os.path.normpath(os.path.join(work_dir, "..", ".."))

base_dir = Path(os.environ.get("BASE_DIR", DEFAULT_BASE_DIR)).resolve()
data_dir = base_dir / "data" / "tep"
model_dir = base_dir / "models"
plot_dir = base_dir / "plots"
results_dir = base_dir / "results"

for _d in (data_dir, model_dir, plot_dir, results_dir):
    os.makedirs(_d, exist_ok=True)

FORCE_RECOMPUTE = False
IS_FINAL = True  

In [ ]:
import thesis_style as ts

TEXTWIDTH_PT = ts.TEXTWIDTH_PT
INCH_PER_PT = ts.INCH_PER_PT
fig_width_in = ts.apply_rc()

In [ ]:
import numpy as np

In [ ]:
# --- Konstanten ---
SEED = 1
RUN_ID = None
ERROR_VARIANT_ID = 29
ERROR_VARIANT_AMP = 1.0
WINDOW_SIZE = 10
ARCH = [5, 5, 1]
ACT = "tanh"
EPOCHS = 200
BATCH = 200
PATIENCE = 20
VAL_SPLIT = 0.15

rng = np.random.default_rng(SEED)

## Daten laden

In [ ]:
import run_registry as rr
data_store = rr.tep_data_store(data_dir)
model_store = rr.tep_model_store(model_dir)
plot_store = rr.tep_plot_store(plot_dir)
res_store = rr.tep_results_store(results_dir)

run_id = data_store.latest_id("data_train") if RUN_ID is None else RUN_ID
if run_id is None:
    raise FileNotFoundError("Kein train-Artefakt. Bitte tep_generation.ipynb ausfuehren.")
train_data = data_store.load("data_train", run_id=run_id, rename=True)
run_cfg = dict(train_data.attrs.get("config", train_data.attrs))
cfg = rr.RunConfig(run_cfg)

ramp_duration = run_cfg["ramp_h"]
soak_time = run_cfg["soak_h"]
test_split = run_cfg["test_split"]

error_variant = {"idv": ERROR_VARIANT_ID, "amp": ERROR_VARIANT_AMP}
test_data = data_store.load("data_test", cfg, rename=True)
error_data = data_store.load("data_error", cfg, variant=error_variant, rename=True)

## Pre-processing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def drop_outliers(df: pd.DataFrame, z_thresh: float=3.0) -> pd.DataFrame:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    z_scores = np.abs((df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std())
    df_out = df.copy()
    for col in numeric_cols:
        outlier_idx = z_scores[col] >= z_thresh
        df_out.loc[outlier_idx, col] = df[col].mean()
    return df_out

Drop outliers

In [ ]:
unchanged_columns = [c for c in train_data.columns if c in ["mode", "original_index", "reset", "status", "time"] or c.startswith("IDV")]
train_outliers_droped = drop_outliers(train_data.drop(columns=unchanged_columns))

Initialize scaler

In [ ]:
scaler = MinMaxScaler()
train_scaled_array = scaler.fit_transform(train_outliers_droped)
train_scaled_data = pd.DataFrame(train_scaled_array, columns=train_outliers_droped.columns, index=train_outliers_droped.index)

Rolling mean

In [ ]:
train_smoothed_data = train_scaled_data.rolling(window=WINDOW_SIZE, min_periods=1).mean()
train_smoothed_data[unchanged_columns] = train_data[unchanged_columns]

Select stationary operations

In [ ]:
n = 5
num_cols = train_data.select_dtypes(include="number").columns
candidates = [c for c in num_cols if c not in ["time", "original_index", "Scenario", "mode", "cost"] and not c.endswith("SP")]
top_n_by_var = train_data[candidates].var().sort_values(ascending=False).head(n).index.tolist()
top_n_by_var

In [ ]:
from scipy.stats import linregress

def window_slope(y:np.ndarray):
    x = np.arange(len(y))
    return linregress(x, y).slope

def mark_consecutive_trues(s:pd.Series, min_len:int):
    return (s.groupby(by=(s != s.shift()).cumsum()).transform('size') >= min_len) * s

def steady_mask(df: pd.DataFrame, win:int, slope_thr:float, min_len: int, offset: int=0):
    roll = df.rolling(win)
    slopes = roll.apply(window_slope, raw=True)
    summed_slopes = slopes.abs().sum(axis=1)
    stationarity = summed_slopes.abs() < slope_thr
    stationarity = mark_consecutive_trues(stationarity, min_len)
    stationarity.iloc[offset:win] = False
    return summed_slopes, stationarity

In [ ]:
def static_steady_mask(df: pd.DataFrame, ramp_duration: float, soak_time: float):
    mask = pd.Series(False, index=df.index)
    for mode in df["mode"].unique():
        mode_data = df[df["mode"] == mode]
        start_time = mode_data["time"].min() + ramp_duration
        end_time = start_time + soak_time - ramp_duration
        mask.loc[mode_data[(mode_data["time"] >= start_time) & (mode_data["time"] <= end_time)].index] = True

    return mask

In [ ]:
train_slopes, train_stationary_mask = steady_mask(train_smoothed_data.loc[:,top_n_by_var], 50, 0.00045, 200, 25)

In [ ]:
import matplotlib.pyplot as plt

filter_start = 0
filter_end = 20000

x_axis = train_smoothed_data["original_index"][filter_start:filter_end] / 20

plt.figure(figsize=(12, 5))
for col in top_n_by_var:
    plt.plot(x_axis, train_smoothed_data[filter_start:filter_end][col], label=col)
plt.xlabel("Time [h]")

in_stationary = False
start_idx = None
for i, val in enumerate(train_stationary_mask[filter_start:filter_end]):
    if val and not in_stationary:
        in_stationary = True
        start_idx = i
    elif not val and in_stationary:
        in_stationary = False
        end_idx = i
        plt.axvspan(x_axis.iloc[start_idx], x_axis.iloc[end_idx-1], color='lightgreen', alpha=0.3)
if in_stationary:
    plt.axvspan(x_axis.iloc[start_idx], x_axis.iloc[-1], color='lightgreen', alpha=0.3)
plt.ylabel("Scaled Value")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("Number of steady points detected:", train_stationary_mask.sum())

In [ ]:
def find_stationary_ranges(df_smoothed, stationary_mask, same_length:bool=True):
    stationary_changes = stationary_mask.astype(int).diff().fillna(0)
    starts = df_smoothed.index[stationary_changes == 1]
    ends = df_smoothed.index[stationary_changes == -1]

    # Edge cases
    if df_smoothed.index[stationary_mask][0] == df_smoothed.index[0]:
        starts = starts.insert(0, df_smoothed.index[0])
    if df_smoothed.index[stationary_mask][-1] == df_smoothed.index[-1]:
        ends = ends.append(pd.Index([df_smoothed.index[-1]]))

    stationary_ranges = list(zip(starts, ends))
    if not same_length:
        return stationary_ranges
    else:
        min_duration = min([end - start for start, end in stationary_ranges])
        stationary_ranges = [(start, start + min_duration) for start, end in stationary_ranges]
        return stationary_ranges

In [ ]:
train_stationary_ranges = find_stationary_ranges(train_smoothed_data, train_stationary_mask, same_length=False)

In [ ]:
stationary_means = pd.concat([train_smoothed_data.loc[(train_smoothed_data["original_index"]>=start) & 
                                      (train_smoothed_data["original_index"]<=end),:].mean(axis=0) 
                                      for start, end in train_stationary_ranges], axis=1).T

In [ ]:
labels_for_costs = [
    'Component A in Purge',
    'Component C in Purge',
    'Component D in Purge',
    'Component E in Purge',
    'Component F in Purge',
    'Component G in Purge',
    'Component H in Purge',
    'Purge Rate',
    'Component D in Product',
    'Component E in Product',
    'Component F in Product',
    'Product Sep Underflow',
    'Compressor Work',
    'Stripper Steam Flow'
]
unchanged_setpoints = [
    'StripLevelSP', 
    'SepLevelSP', 
    'ReactorLevelSP', 
    'ReactorPressSP', 
    'SteamValvePosSP', 
    'AgitatorSpeedSP'
]

In [ ]:
import seaborn as sns

corr = stationary_means.corr()

setpoint_labels_update = [col for col in stationary_means.columns if col not in unchanged_setpoints and col not in labels_for_costs]
main_corr = corr.loc["cost", setpoint_labels_update].abs().sort_values(ascending=False)
main_corr.head(15)

## Heatmap

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(corr.loc[main_corr[:15].index, main_corr[:15].index], cmap="coolwarm", annot=False, fmt=".2f")
plt.tight_layout()
plt.show()

Prepare training

In [ ]:
feature_cols = main_corr.iloc[1:8].index
excluded_cols = ["Stripper Temp", "Product Sep Temp"]
feature_cols = [col for col in feature_cols if col not in labels_for_costs and col not in excluded_cols]
len(feature_cols), feature_cols

In [ ]:
train_stationary_only = True

if train_stationary_only:
    train_preprocessed_data = pd.concat([train_smoothed_data.loc[(train_smoothed_data["original_index"]>=start) & 
                                      (train_smoothed_data["original_index"]<=end),:]
                                      for start, end in train_stationary_ranges])
else:
    train_preprocessed_data = train_smoothed_data.copy()

In [ ]:
from sklearn.model_selection import train_test_split

val_split = 0.15

x_temp = train_preprocessed_data.loc[:, feature_cols].values
y_temp = train_preprocessed_data.loc[:, "cost"].values

x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=val_split, random_state=SEED)

print(f"Train shape: {x_train.shape}, {y_train.shape}")
print(f"Validation shape: {x_val.shape}, {y_val.shape}")

## Train Neural network

In [ ]:
model_params = {"arch": ARCH, "act": ACT, "opt": "adam", "epochs": EPOCHS,
                "batch": BATCH, "patience": PATIENCE, "val_split": VAL_SPLIT,
                "seed": SEED, "window_size": WINDOW_SIZE, "features": list(feature_cols)}

model_cfg = cfg.compose(model=model_params)

_model_loaded_from_cache = model_store.exists("model", model_cfg) and not FORCE_RECOMPUTE

_scored_loaded_from_cache = (
    data_store.exists("data_scored_test", model_cfg)
    and data_store.exists("data_scored_error", model_cfg, variant=error_variant)
    and not FORCE_RECOMPUTE
)

In [ ]:
if _model_loaded_from_cache:
    _mc = model_store.load("model", model_cfg, rename=True)
    model, feature_cols = _mc["model"], _mc["feature_cols"]
else:
    import tensorflow as tf
    from tensorflow import keras
    from keras import layers
    from keras.callbacks import EarlyStopping

    tf.keras.utils.set_random_seed(SEED)

    model = keras.Sequential([
        layers.Dense(ARCH[0], activation=ACT, input_shape=(len(feature_cols),)),
        layers.Dense(ARCH[1], activation=ACT), 
        layers.Dense(ARCH[2])
    ])

    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    early_stop = EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)

    history = model.fit(
        x_train, y_train, 
        validation_data=(x_val, y_val),
        epochs=EPOCHS, 
        batch_size=BATCH, 
        callbacks=[early_stop], 
        verbose=1
    )
    model_store.save({"model": model, "feature_cols": list(feature_cols), "window_size": WINDOW_SIZE},
               "model", model_cfg)

In [ ]:
if not _model_loaded_from_cache:
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(history.history['mae'], label='Train MAE')
    plt.plot(history.history['val_mae'], label='Val MAE')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Preprocess test data

Wird uebersprungen und aus dem Cache geladen, falls `data_scored_test` fuer diese `model_cfg` bereits existiert (analog zum Modell-Cache).

In [ ]:
if _scored_loaded_from_cache:
    test_preprocessed_data = data_store.load("data_scored_test", model_cfg, rename=True)
    print(f"Scored-Testdaten aus Cache geladen ({len(test_preprocessed_data)} Zeilen)")

In [ ]:
if not _scored_loaded_from_cache:
    test_outliers_droped = drop_outliers(test_data.drop(columns=unchanged_columns))

In [ ]:
if not _scored_loaded_from_cache:
    test_scaled_array = scaler.transform(test_outliers_droped)
    test_scaled_data = pd.DataFrame(test_scaled_array, columns=test_outliers_droped.columns, index=test_outliers_droped.index)

In [ ]:
if not _scored_loaded_from_cache:
    test_smoothed_data = test_scaled_data.rolling(window=WINDOW_SIZE, min_periods=1).mean()
    test_smoothed_data[unchanged_columns] = test_data[unchanged_columns]

In [ ]:
if not _scored_loaded_from_cache:
    test_stationary_mask = static_steady_mask(test_smoothed_data, ramp_duration, soak_time)
    test_stationary_ranges = find_stationary_ranges(test_smoothed_data, test_stationary_mask, same_length=False)

In [ ]:
if not _scored_loaded_from_cache:
    test_stationary_only = False

    if test_stationary_only:
        test_preprocessed_data = pd.concat([test_smoothed_data.loc[(test_smoothed_data["original_index"]>=start) & 
                                          (test_smoothed_data["original_index"]<=end),:]
                                          for start, end in test_stationary_ranges])
    else:
        test_preprocessed_data = test_smoothed_data.copy()

Add model predictions and model error to test data

In [ ]:
if not _scored_loaded_from_cache:
    test_features = test_preprocessed_data[feature_cols].to_numpy(dtype="float32", copy=False)
    y_pred = model.predict(test_features)

    test_preprocessed_data.loc[:, "y_pred"] = y_pred
    test_preprocessed_data.loc[:, "model_error"] = test_preprocessed_data["y_pred"] - test_preprocessed_data["cost"]

    test_preprocessed_data.set_index("original_index", inplace=True)

In [ ]:
if not _scored_loaded_from_cache:
    test_reset_indices = test_smoothed_data[test_smoothed_data["reset"] == True].index

In [ ]:
if not _scored_loaded_from_cache:
    test_reset_indices

In [ ]:
if not _scored_loaded_from_cache:
    test_data_sampled = test_preprocessed_data.sample(frac=.1, random_state=SEED)

    plt.figure(figsize=(12, 5))
    plt.scatter(test_data_sampled.index, test_data_sampled["model_error"], label="Model error", s=10)

    for i, reset_idx in enumerate(test_reset_indices):
        plt.axvline(reset_idx, color='red', linestyle='--', alpha=0.6, label='reset' if i == 0 else "")

    plt.xlabel("Time [h]")
    plt.ylabel("Model error")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
if not _scored_loaded_from_cache:
    data_store.save(test_preprocessed_data, "data_scored_test", model_cfg,
               meta={"feature_cols": list(feature_cols),
                     "stationary_ranges": [(int(s), int(e)) for s, e in test_stationary_ranges]})

## Preprocess error data

Wird uebersprungen und aus dem Cache geladen, falls `data_scored_error` fuer diese `model_cfg` bereits existiert.

In [ ]:
if _scored_loaded_from_cache:
    error_preprocessed_data = data_store.load("data_scored_error", model_cfg, variant=error_variant, rename=True)
    print(f"Scored-Fehlerdaten aus Cache geladen ({len(error_preprocessed_data)} Zeilen)")

In [ ]:
if not _scored_loaded_from_cache:
    error_outliers_droped = drop_outliers(error_data.drop(columns=unchanged_columns))

In [ ]:
if not _scored_loaded_from_cache:
    error_scaled_array = scaler.transform(error_outliers_droped)
    error_scaled_data = pd.DataFrame(error_scaled_array, columns=error_outliers_droped.columns, index=error_outliers_droped.index)

In [ ]:
if not _scored_loaded_from_cache:
    error_smoothed_data = error_scaled_data.rolling(window=WINDOW_SIZE, min_periods=1).mean()
    error_smoothed_data[unchanged_columns] = error_data[unchanged_columns]

In [ ]:
if not _scored_loaded_from_cache:
    error_slopes, error_stationary_mask = steady_mask(error_smoothed_data.loc[:,top_n_by_var], 50, 0.00045, 200, 25)
    error_stationary_ranges = find_stationary_ranges(error_smoothed_data, error_stationary_mask, same_length=False)

In [ ]:
if not _scored_loaded_from_cache:
    error_stationary_only = test_stationary_only

    if error_stationary_only:
        error_preprocessed_data = pd.concat([error_smoothed_data.loc[(error_smoothed_data["original_index"]>=start) & 
                                          (error_smoothed_data["original_index"]<=end),:]
                                          for start, end in test_stationary_ranges])
    else:
        error_preprocessed_data = error_smoothed_data.copy()

Add model predictions and model error to error data

In [ ]:
if not _scored_loaded_from_cache:
    error_features = error_preprocessed_data[feature_cols].to_numpy(dtype="float32", copy=False)
    y_pred = model.predict(error_features)

    error_preprocessed_data.loc[:, "y_pred"] = y_pred
    error_preprocessed_data.loc[:, "model_error"] = error_preprocessed_data["y_pred"] - error_preprocessed_data["cost"]

    error_preprocessed_data.set_index("original_index", inplace=True)

In [ ]:
if not _scored_loaded_from_cache:
    error_reset_indices = error_smoothed_data[error_smoothed_data["reset"] == True].index

In [ ]:
if not _scored_loaded_from_cache:
    plt.figure(figsize=(12, 5))
    plt.plot(error_preprocessed_data["cost"], label="y_test (True)", alpha=0.5, linewidth=3, color="green")
    plt.plot(error_preprocessed_data["y_pred"], label="y_pred (Predicted)", alpha=0.5, color="orange")
    plt.scatter(error_preprocessed_data.index, error_preprocessed_data["model_error"], label="model_error", s=3)

    for i, reset_idx in enumerate(error_reset_indices):
        plt.axvline(reset_idx, color='red', linestyle='--', alpha=0.6, label='reset' if i == 0 else "")

    plt.xlabel("Sample Index")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
if not _scored_loaded_from_cache:
    data_store.save(error_preprocessed_data, "data_scored_error", model_cfg,
               variant=error_variant, meta={"idv": 29})

## Vergleich Modellfehler: nominal vs. Fehlerfall

In [ ]:
combined = []

for mode in test_preprocessed_data["mode"].unique():
    test_temp = test_preprocessed_data.loc[test_preprocessed_data["mode"]==mode, ["model_error", "reset"]]
    test_temp.reset_index(inplace=True, drop=True)
    fault_temp = error_preprocessed_data.loc[error_preprocessed_data["mode"]==mode, ["model_error", "reset"]]
    fault_temp.reset_index(inplace=True, drop=True)
    combined.append(pd.concat([test_temp, fault_temp], axis=1))
combined_data = pd.concat(combined, axis=0, ignore_index=True)
combined_data.columns = ["model_error_normal", "reset_normal", "model_error_fault", "reset_fault"]

In [ ]:
combined_data["time"] = combined_data.index / 20
sampled_combined = combined_data.sample(frac=0.1, random_state=SEED)
sampled_combined.sort_index(inplace=True)

In [ ]:
from matplotlib.lines import Line2D

ALPHA, S = 0.5, 6

fig, ax = plt.subplots(figsize=(fig_width_in, fig_width_in * 0.45))
ax.scatter(sampled_combined["time"], sampled_combined["model_error_normal"],
           s=S, alpha=ALPHA, color=ts.C["drift_free"], edgecolors="none", rasterized=True)
ax.scatter(sampled_combined["time"], sampled_combined["model_error_fault"],
           s=S, alpha=ALPHA, color=ts.C["drift_afflicted"], edgecolors="none", rasterized=True)

proxies = [Line2D([0], [0], marker='o', linestyle='', color=ts.C["drift_free"], label="Modellfehler (nominal)"),
           Line2D([0], [0], marker='o', linestyle='', color=ts.C["drift_afflicted"], label=f"Modellfehler (IDV {ERROR_VARIANT_ID})")]
for p in proxies:
    p.set_alpha(ALPHA)
ax.legend(handles=proxies, loc="lower left", frameon=False)

ax.set_xlabel("Zeit $t$ [h]")
ax.set_ylabel("Modellfehler")
ax.margins(x=0)
ax.grid(True, alpha=0.2)
fig.tight_layout()
if not _model_loaded_from_cache:
    plot_store.save_figure(fig, "error", model_cfg, final=IS_FINAL, savefig_kwargs={"dpi": 300}, archive_kwargs={"dpi": 300}, variant=error_variant)
plt.show()

## Numerische Auswertung der Modelldegradation

In [ ]:
r0 = test_preprocessed_data["model_error"].to_numpy(dtype=float)
r1 = error_preprocessed_data["model_error"].to_numpy(dtype=float)

stats = pd.DataFrame({
    "MLP": dict(
        rmse_free=np.sqrt(np.nanmean(r0**2)),
        rmse_fault=np.sqrt(np.nanmean(r1**2)),
        std_free=np.nanstd(r0),
        bias_fault=np.nanmean(r1),
        std_fault=np.nanstd(r1),
    )
}).T
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
print(stats.to_string())

## Ergebnisse speichern

In [ ]:
from src.utils import results_export as rx

_keys = [(k, k, "num") for k in
         ["rmse_free", "rmse_fault", "std_free", "bias_fault", "std_fault"]]

(rx.ResultDoc()
   .integer("n_test", len(test_preprocessed_data))
   .integer("n_error", len(error_preprocessed_data))
   .stats(stats, _keys, into="models")
   .save(res_store, "error", model_cfg, final=IS_FINAL, variant=error_variant))

In [ ]:
def _de(x, nd):
    if x is None or (isinstance(x, float) and x != x):
        return "--"
    return f"{x:.{nd}f}".replace(".", "{,}")

_keys = [("rmse_free", "RmseFree", 3), ("rmse_fault", "RmseFault", 3),
         ("std_free", "SigmaFree", 3), ("bias_fault", "Bias", 3), ("std_fault", "Sigma", 3)]
_lines = ["% automatisch erzeugt aus tep_model.ipynb - nicht manuell editieren"]
for m in stats.index:
    for col, suf, nd in _keys:
        _lines.append(rf"\newcommand{{\tepstat{m}{suf}}}{{{_de(stats.loc[m, col], nd)}}}")
_tex = "\n".join(_lines) + "\n"
_pt, _ = res_store.save_text(_tex, "tep_model_error_stats", model_cfg, final=IS_FINAL, variant=error_variant)
print(f"LaTeX-Makros -> {_pt}\n")
print(_tex)